# Li (2017) 3D CNN for Hyperspectral Image Classification - CORRECTED

## Implementation Notes
This notebook faithfully recreates the architecture from:
*Li, Y., Zhang, H., & Shen, Q. (2017). Spectral–spatial classification of hyperspectral imagery with 3D convolutional neural network. Remote Sensing, 9(1), 67.*

### Specified Parameters (from paper):
- Window size: 5×5
- **C1 filters: 2** (3×3×7 kernel)
- **C2 filters: 4** (3×3×3 kernel)
- Momentum: 0.9
- Weight decay: 0.0005
- Iterations: 100,000 (Indian Pines), 8,000 (Botswana)
- Train/test split: 50-50
- Validation: 10 repetitions, averaged

### ⚠️ ESTIMATED Parameters (not specified in paper):
- **Learning rate: 0.01** (typical for SGD, based on common practice)
- **Batch size: 64** (reasonable for this dataset size)
- **FC layer neurons: 128** (common choice for shallow networks)
- **Weight initialization: Glorot uniform** (Keras default)

These estimated values are reasonable defaults but may differ from the original implementation.

In [1]:
# !pip install tensorflow-gpu
# !conda create --name tf-gpu tensorflow-gpu
# !conda activate tf-gpu

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"Built with CUDA: {tf.test.is_built_with_cuda()}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

2026-02-25 10:55:18.434612: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.20.0
Built with CUDA: True
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Load HSI Datasets

Download and load Indian Pines and Pavia University datasets.

In [2]:
import os
import urllib.request
import scipy.io as sio
import numpy as np

def download_file(url, filename):
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
        print(f"Downloaded {filename}")
    else:
        print(f"{filename} already exists.")

# Define data directory and base URL
data_dir = '../hsi_datasets/hsi_researches'
base_url = 'http://www.ehu.eus/ccwintco/uploads/'

# Create data directory
os.makedirs(data_dir, exist_ok=True)

# Dataset information
datasets_info = {
    'indian_pines': {
        'image_file': 'Indian_pines_corrected.mat',
        'gt_file': 'Indian_pines_gt.mat',
        'image_url_suffix': '6/67/Indian_pines_corrected.mat',
        'gt_url_suffix': 'c/c4/Indian_pines_gt.mat',
        'image_var': 'indian_pines_corrected',
        'gt_var': 'indian_pines_gt'
    },
    # 'pavia_university': {
    #     'image_file': 'PaviaU.mat',
    #     'gt_file': 'PaviaU_gt.mat',
    #     'image_url_suffix': 'e/ee/PaviaU.mat',
    #     'gt_url_suffix': '5/50/PaviaU_gt.mat',
    #     'image_var': 'paviaU',
    #     'gt_var': 'paviaU_gt'
    # }
}

# Download and load datasets
indian_pines_image = None
indian_pines_gt = None
# pavia_university_image = None
# pavia_university_gt = None

for dataset_name, info in datasets_info.items():
    image_path = os.path.join(data_dir, info['image_file'])
    gt_path = os.path.join(data_dir, info['gt_file'])

    # Download files
    download_file(base_url + info['image_url_suffix'], image_path)
    download_file(base_url + info['gt_url_suffix'], gt_path)

    # Load files
    print(f"Loading {info['image_file']}...")
    data = sio.loadmat(image_path)
    globals()[f'{dataset_name}_image'] = data[info['image_var']]
    
    print(f"Loading {info['gt_file']}...")
    gt_data = sio.loadmat(gt_path)
    globals()[f'{dataset_name}_gt'] = gt_data[info['gt_var']]

# Display shapes
print("\n--- Dataset Shapes ---")
print(f"Indian Pines Image: {indian_pines_image.shape}")
print(f"Indian Pines GT: {indian_pines_gt.shape}")
# print(f"Pavia University Image: {pavia_university_image.shape}")
# print(f"Pavia University GT: {pavia_university_gt.shape}")

../hsi_datasets/hsi_researches/Indian_pines_corrected.mat already exists.
../hsi_datasets/hsi_researches/Indian_pines_gt.mat already exists.
Loading Indian_pines_corrected.mat...
Loading Indian_pines_gt.mat...

--- Dataset Shapes ---
Indian Pines Image: (145, 145, 200)
Indian Pines GT: (145, 145)


## 2. Data Preprocessing and Patch Extraction

Apply mirror padding, filter unlabeled samples, normalize, and extract 5×5×L patches.

In [3]:
def preprocess_hsi_data(hsi_image, gt_image, patch_size=5, padding_amount=2):
    """
    Preprocess HSI data as per Li (2017):
    - Mirror padding for border pixels
    - Extract 5×5×L patches for labeled pixels
    - Normalize by max value
    """
    # Apply mirror padding
    padded_hsi = np.pad(
        hsi_image, 
        ((padding_amount, padding_amount), 
         (padding_amount, padding_amount), 
         (0, 0)), 
        mode='reflect'
    )

    patches = []
    labels = []

    h, w, l = hsi_image.shape

    # Extract patches for labeled pixels only
    for r in range(h):
        for c in range(w):
            if gt_image[r, c] > 0:  # Labeled sample
                # Extract 5×5×L patch from padded image
                padded_r = r + padding_amount
                padded_c = c + padding_amount

                patch = padded_hsi[
                    padded_r - padding_amount : padded_r + padding_amount + 1,
                    padded_c - padding_amount : padded_c + padding_amount + 1,
                    :
                ]
                patches.append(patch)
                # Convert to 0-indexed labels
                labels.append(gt_image[r, c] - 1)

    # Convert to arrays
    patches = np.array(patches)
    labels = np.array(labels)

    # Normalize by max value
    patches = patches / hsi_image.max()

    return patches, labels

print("Processing datasets...")

# Process Indian Pines
indian_pines_patches, indian_pines_labels = preprocess_hsi_data(
    indian_pines_image, indian_pines_gt
)
print(f"Indian Pines - Patches: {indian_pines_patches.shape}, Labels: {indian_pines_labels.shape}")

# # Process Pavia University
# pavia_university_patches, pavia_university_labels = preprocess_hsi_data(
#     pavia_university_image, pavia_university_gt
# )
# print(f"Pavia University - Patches: {pavia_university_patches.shape}, Labels: {pavia_university_labels.shape}")

Processing datasets...
Indian Pines - Patches: (10249, 5, 5, 200), Labels: (10249,)


## 3. Define 3D CNN Model Architecture

### Architecture (Li 2017):
```
Input (5×5×L×1)
  ↓
Conv3D: 2 filters, 3×3×7 kernel, ReLU
  ↓
Conv3D: 4 filters, 3×3×3 kernel, ReLU (implied)
  ↓
Flatten
  ↓
Dense: [ESTIMATED: 128 units], ReLU
  ↓
Dense: num_classes, Softmax
```

In [5]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv3D, Flatten, Dense
from tensorflow.keras.models import Model

def create_3d_cnn_model(input_shape, num_classes, fc_units=128):
    """
    Create 3D CNN model as per Li (2017).
    
    SPECIFIED PARAMETERS:
    - C1: 2 filters, 3×3×7 kernel
    - C2: 4 filters, 3×3×3 kernel
    
    ESTIMATED PARAMETERS:
    - fc_units: 128 (not specified in paper)
    - Weight initialization: Glorot uniform (Keras default)
    """
    input_layer = Input(shape=input_shape)

    # C1: 2 filters, 3×3×7 kernel (SPECIFIED)
    conv1 = Conv3D(
        filters=2,  # PAPER SPECIFIED
        kernel_size=(3, 3, 7),  # PAPER SPECIFIED
        activation='relu',
        padding='same'
    )(input_layer)

    # C2: 4 filters, 3×3×3 kernel (SPECIFIED)
    conv2 = Conv3D(
        filters=4,  # PAPER SPECIFIED
        kernel_size=(3, 3, 3),  # PAPER SPECIFIED
        activation='relu',
        padding='same'
    )(conv1)

    # Flatten
    flatten = Flatten()(conv2)

    # Fully connected layer (ESTIMATED: paper doesn't specify units)
    dense1 = Dense(
        units=fc_units,  # ESTIMATED - not in paper
        activation='relu'
    )(flatten)

    # Output layer
    output_layer = Dense(
        units=num_classes,
        activation='softmax'
    )(dense1)

    model = Model(inputs=input_layer, outputs=output_layer)
    return model

print("Model architecture defined.")

Model architecture defined.


## 4. Training Configuration

### SPECIFIED Parameters:
- Momentum: 0.9
- Weight decay: 0.0005
- Iterations: 100,000
- Repetitions: 10
- Train/test split: 50-50

### ⚠️ ESTIMATED Parameters:
- **Learning rate: 0.01** (not in paper, typical for SGD)
- **Batch size: 64** (not in paper, reasonable for dataset)

In [6]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import SGD

# SPECIFIED PARAMETERS
num_repetitions = 10  # PAPER SPECIFIED
total_iterations = 100000  # PAPER SPECIFIED (Indian Pines)
momentum = 0.9  # PAPER SPECIFIED
weight_decay = 0.0005  # PAPER SPECIFIED
test_size = 0.5  # PAPER SPECIFIED (50-50 split)

# ESTIMATED PARAMETERS (not in paper)
learning_rate = 0.01  # ⚠️ ESTIMATED
batch_size = 64  # ⚠️ ESTIMATED
fc_units = 128  # ⚠️ ESTIMATED

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
print("\nSPECIFIED Parameters (from paper):")
print(f"  Repetitions: {num_repetitions}")
print(f"  Total iterations: {total_iterations}")
print(f"  Momentum: {momentum}")
print(f"  Weight decay: {weight_decay}")
print(f"  Train/test split: {int((1-test_size)*100)}/{int(test_size*100)}")
print(f"  C1 filters: 2")
print(f"  C2 filters: 4")
print("\n⚠️  ESTIMATED Parameters (not in paper):")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  FC layer units: {fc_units}")
print("=" * 60)

TRAINING CONFIGURATION

SPECIFIED Parameters (from paper):
  Repetitions: 10
  Total iterations: 100000
  Momentum: 0.9
  Weight decay: 0.0005
  Train/test split: 50/50
  C1 filters: 2
  C2 filters: 4

⚠️  ESTIMATED Parameters (not in paper):
  Learning rate: 0.01
  Batch size: 64
  FC layer units: 128


## 5. Train and Evaluate Indian Pines Model

Perform 10 repetitions with independent train/test splits.

In [7]:
# Get number of classes
num_classes_indian_pines = len(np.unique(indian_pines_labels))
print(f"Number of classes (Indian Pines): {num_classes_indian_pines}")

# Results storage
indian_pines_results = {
    'accuracies': [],
    'losses': []
}

print(f"\nStarting training for Indian Pines ({num_repetitions} repetitions)...\n")

for i in range(num_repetitions):
    print(f"{'='*60}")
    print(f"Repetition {i+1}/{num_repetitions}")
    print(f"{'='*60}")

    # Clear session and create fresh model
    tf.keras.backend.clear_session()

    # Create new train/test split (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        indian_pines_patches,
        indian_pines_labels,
        test_size=test_size,
        stratify=indian_pines_labels,
        random_state=42 + i  # Different seed per repetition
    )

    # Reshape to add channel dimension
    X_train = np.expand_dims(X_train, -1)
    X_test = np.expand_dims(X_test, -1)

    # One-hot encode labels
    y_train_one_hot = tf.keras.utils.to_categorical(y_train, num_classes=num_classes_indian_pines)
    y_test_one_hot = tf.keras.utils.to_categorical(y_test, num_classes=num_classes_indian_pines)

    # Create model
    input_shape = X_train.shape[1:]  # (5, 5, 200, 1)
    model = create_3d_cnn_model(input_shape, num_classes_indian_pines, fc_units=fc_units)

    # Configure optimizer with SPECIFIED parameters
    optimizer = SGD(
        learning_rate=learning_rate,  # ESTIMATED
        momentum=momentum,  # PAPER SPECIFIED
        weight_decay=weight_decay  # PAPER SPECIFIED (renamed from decay)
    )

    # Compile model
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Calculate epochs needed for total_iterations
    steps_per_epoch = max(1, len(X_train) // batch_size)
    num_epochs = int(np.ceil(total_iterations / steps_per_epoch))

    print(f"Training for {num_epochs} epochs ({steps_per_epoch} steps/epoch)...")

    # Train model
    history = model.fit(
        X_train, y_train_one_hot,
        epochs=num_epochs,
        batch_size=batch_size,
        validation_data=(X_test, y_test_one_hot),
        verbose=0
    )

    # Evaluate on test set
    loss, accuracy = model.evaluate(X_test, y_test_one_hot, verbose=0)
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Loss: {loss:.4f}\n")

    # Store results
    indian_pines_results['accuracies'].append(accuracy)
    indian_pines_results['losses'].append(loss)

# Calculate summary statistics
print("\n" + "="*60)
print("INDIAN PINES - SUMMARY STATISTICS")
print("="*60)
print(f"Mean Accuracy: {np.mean(indian_pines_results['accuracies']):.4f} ± {np.std(indian_pines_results['accuracies']):.4f}")
print(f"Mean Loss: {np.mean(indian_pines_results['losses']):.4f} ± {np.std(indian_pines_results['losses']):.4f}")
print(f"\nAccuracies: {[f'{a:.4f}' for a in indian_pines_results['accuracies']]}")
print(f"Losses: {[f'{l:.4f}' for l in indian_pines_results['losses']]}")

Number of classes (Indian Pines): 16

Starting training for Indian Pines (10 repetitions)...

Repetition 1/10


I0000 00:00:1771988319.755519  198098 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Ti, pci bus id: 0000:09:00.0, compute capability: 8.6


Training for 1250 epochs (80 steps/epoch)...


2026-02-25 10:58:41.549573: I external/local_xla/xla/service/service.cc:163] XLA service 0x70d040004c90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-25 10:58:41.549602: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3080 Ti, Compute Capability 8.6
2026-02-25 10:58:41.588656: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-25 10:58:41.676053: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
2026-02-25 10:58:42.216671: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_431', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1771988323.801336  198790 device_compiler.h:196] Compiled cluster using XLA!  This line is logged a

Test Accuracy: 0.9385
Test Loss: 0.2736

Repetition 2/10
Training for 1250 epochs (80 steps/epoch)...
Test Accuracy: 0.9508
Test Loss: 0.2645

Repetition 3/10
Training for 1250 epochs (80 steps/epoch)...
Test Accuracy: 0.9649
Test Loss: 0.1727

Repetition 4/10
Training for 1250 epochs (80 steps/epoch)...
Test Accuracy: 0.8451
Test Loss: 0.6422

Repetition 5/10
Training for 1250 epochs (80 steps/epoch)...
Test Accuracy: 0.9512
Test Loss: 0.2724

Repetition 6/10
Training for 1250 epochs (80 steps/epoch)...
Test Accuracy: 0.9471
Test Loss: 0.2667

Repetition 7/10
Training for 1250 epochs (80 steps/epoch)...
Test Accuracy: 0.9526
Test Loss: 0.2621

Repetition 8/10
Training for 1250 epochs (80 steps/epoch)...
Test Accuracy: 0.9662
Test Loss: 0.1384

Repetition 9/10
Training for 1250 epochs (80 steps/epoch)...
Test Accuracy: 0.9666
Test Loss: 0.1579

Repetition 10/10
Training for 1250 epochs (80 steps/epoch)...
Test Accuracy: 0.2396
Test Loss: 2.3257


INDIAN PINES - SUMMARY STATISTICS
Mean 

## 6. Train and Evaluate Pavia University Model

Same procedure for Pavia University dataset.

In [ ]:
# Get number of classes
num_classes_pavia = len(np.unique(pavia_university_labels))
print(f"Number of classes (Pavia University): {num_classes_pavia}")

# Results storage
pavia_results = {
    'accuracies': [],
    'losses': []
}

print(f"\nStarting training for Pavia University ({num_repetitions} repetitions)...\n")

for i in range(num_repetitions):
    print(f"{'='*60}")
    print(f"Repetition {i+1}/{num_repetitions}")
    print(f"{'='*60}")

    # Clear session and create fresh model
    tf.keras.backend.clear_session()

    # Create new train/test split (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        pavia_university_patches,
        pavia_university_labels,
        test_size=test_size,
        stratify=pavia_university_labels,
        random_state=42 + i
    )

    # Reshape to add channel dimension
    X_train = np.expand_dims(X_train, -1)
    X_test = np.expand_dims(X_test, -1)

    # One-hot encode labels
    y_train_one_hot = tf.keras.utils.to_categorical(y_train, num_classes=num_classes_pavia)
    y_test_one_hot = tf.keras.utils.to_categorical(y_test, num_classes=num_classes_pavia)

    # Create model
    input_shape = X_train.shape[1:]  # (5, 5, 103, 1)
    model = create_3d_cnn_model(input_shape, num_classes_pavia, fc_units=fc_units)

    # Configure optimizer
    optimizer = SGD(
        learning_rate=learning_rate,
        momentum=momentum,
        weight_decay=weight_decay
    )

    # Compile model
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Calculate epochs
    steps_per_epoch = max(1, len(X_train) // batch_size)
    num_epochs = int(np.ceil(total_iterations / steps_per_epoch))

    print(f"Training for {num_epochs} epochs ({steps_per_epoch} steps/epoch)...")

    # Train model
    history = model.fit(
        X_train, y_train_one_hot,
        epochs=num_epochs,
        batch_size=batch_size,
        validation_data=(X_test, y_test_one_hot),
        verbose=0
    )

    # Evaluate on test set
    loss, accuracy = model.evaluate(X_test, y_test_one_hot, verbose=0)
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Loss: {loss:.4f}\n")

    # Store results
    pavia_results['accuracies'].append(accuracy)
    pavia_results['losses'].append(loss)

# Calculate summary statistics
print("\n" + "="*60)
print("PAVIA UNIVERSITY - SUMMARY STATISTICS")
print("="*60)
print(f"Mean Accuracy: {np.mean(pavia_results['accuracies']):.4f} ± {np.std(pavia_results['accuracies']):.4f}")
print(f"Mean Loss: {np.mean(pavia_results['losses']):.4f} ± {np.std(pavia_results['losses']):.4f}")
print(f"\nAccuracies: {[f'{a:.4f}' for a in pavia_results['accuracies']]}")
print(f"Losses: {[f'{l:.4f}' for l in pavia_results['losses']]}")

## 7. Final Summary

Compare results across both datasets.

In [ ]:
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print("\nIndian Pines:")
print(f"  Mean Accuracy: {np.mean(indian_pines_results['accuracies']):.4f} ± {np.std(indian_pines_results['accuracies']):.4f}")
print(f"  Mean Loss: {np.mean(indian_pines_results['losses']):.4f} ± {np.std(indian_pines_results['losses']):.4f}")

print("\nPavia University:")
print(f"  Mean Accuracy: {np.mean(pavia_results['accuracies']):.4f} ± {np.std(pavia_results['accuracies']):.4f}")
print(f"  Mean Loss: {np.mean(pavia_results['losses']):.4f} ± {np.std(pavia_results['losses']):.4f}")

print("\n" + "="*60)
print("IMPLEMENTATION NOTES")
print("="*60)
print("\n✓ FAITHFULLY IMPLEMENTED:")
print("  - C1: 2 filters (3×3×7)")
print("  - C2: 4 filters (3×3×3)")
print("  - Momentum: 0.9")
print("  - Weight decay: 0.0005")
print("  - 100,000 iterations")
print("  - 50-50 train/test split")
print("  - 10 repetitions with averaging")
print("  - Mirror padding")
print("  - 5×5 spatial patches")

print("\n⚠️  ESTIMATED (not in paper):")
print("  - Learning rate: 0.01")
print("  - Batch size: 64")
print("  - FC layer: 128 units")
print("  - Weight init: Glorot uniform")
print("="*60)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy comparison
axes[0].boxplot([indian_pines_results['accuracies'], pavia_results['accuracies']], 
                labels=['Indian Pines', 'Pavia University'])
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Test Accuracy Distribution (10 repetitions)')
axes[0].grid(True, alpha=0.3)

# Loss comparison
axes[1].boxplot([indian_pines_results['losses'], pavia_results['losses']], 
                labels=['Indian Pines', 'Pavia University'])
axes[1].set_ylabel('Loss')
axes[1].set_title('Test Loss Distribution (10 repetitions)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()